# ImageCubes API Demo

Interactive notebook to work with the ImageCubes API for creating Sentinel-2 datacubes and computing NDVI.

### API Endpoints

| Method | Endpoint | Description |
|--------|----------|-------------|
| `POST` | `/api/v1/imagecubes` | Create a new cube |
| `GET` | `/api/v1/imagecubes` | List all cubes |
| `GET` | `/api/v1/imagecubes/{job_id}` | Get cube status/details |
| `GET` | `/api/v1/imagecubes/{job_id}/plan` | Get execution plan |
| `GET` | `/api/v1/imagecubes/{job_id}/artifact-url` | Get signed Zarr URL |
| `GET` | `/api/v1/imagecubes/{job_id}/tasks` | List tasks for a job |
| `PATCH` | `/api/v1/imagecubes/{job_id}` | Cancel (`status: failed`) or resume (`status: dispatching`) |
| `POST` | `/api/v1/imagecubes/{job_id}/tasks/{task_id}/retry` | Retry a failed task |
| `POST` | `/api/v1/imagecubes/{job_id}/runs` | Trigger a new run |
| `DELETE` | `/api/v1/imagecubes/{job_id}` | Delete cube (feature-flagged) |

In [12]:
import requests
import json
import time
import os
from dotenv import load_dotenv

# Load env
load_dotenv("../.env")

API_BASE = "https://imagecube-toucan-api.jollyground-fbd4d1eb.eastus.azurecontainerapps.io"
TOKEN = os.getenv("IMAGECUBES_TOKEN", "")

headers = {"Authorization": f"Bearer {TOKEN}"}
print(f"Token loaded: {'Yes' if TOKEN else 'No (run image_cube.py to authenticate)'}")

Token loaded: Yes


## List Existing Cubes

In [15]:
resp = requests.get(f"{API_BASE}/api/v1/imagecubes", headers=headers)
resp.raise_for_status()
cubes = resp.json()

if isinstance(cubes, list):
    items = cubes
else:
    items = cubes.get("items", [])

print(f"Found {len(items)} cubes:\n")
for c in items[:10]:
    print(f"  {c['id']} | {c.get('name','?')} | status={c.get('status')} | bbox={c.get('bbox')}")

Found 0 cubes:



## Create a New Cube

Define the area, date range, and bands to include.

In [16]:
# Define parameters
bbox = [-9.5, 38.6, -9.0, 39.0]  # Lisbon area
start_date = "2024-06-01"
end_date = "2024-08-31"

payload = {
    "bbox": bbox,
    "start_date": start_date,
    "end_date": end_date,
    "interval": "biweekly",
    "bands": ["B04", "B08"],  # Red + NIR for NDVI
    "cloud_cover_threshold": 20,
    "blend_method": "median",
    "cloud_mask_mode": "cloud_masked",
    "name": f"demo_lisbon_{start_date}",
    "description": "Demo NDVI cube for Lisbon area",
}

resp = requests.post(f"{API_BASE}/api/v1/imagecubes", json=payload, headers=headers)
if resp.status_code == 422:
    print(f"Validation error (422):\n{json.dumps(resp.json(), indent=2)}")
else:
    resp.raise_for_status()
    cube = resp.json()
    cube_id = cube["id"]
    print(f"Created cube: {cube_id}")
    print(json.dumps(cube, indent=2))

KeyError: 'id'

## Poll for Completion

In [ ]:
# Poll until done (or use an existing cube_id)
# cube_id = "<paste-id-here>"  # uncomment to use an existing cube

max_wait = 300  # seconds
elapsed = 0
interval = 5

while elapsed < max_wait:
    resp = requests.get(f"{API_BASE}/api/v1/imagecubes/{cube_id}", headers=headers)
    resp.raise_for_status()
    info = resp.json()
    status = info.get("status", "unknown")
    print(f"  [{elapsed}s] Status: {status}")
    
    if status == "completed":
        print("\nCube is ready!")
        break
    elif status in ("failed", "error"):
        print(f"\nCube failed: {info.get('error')}")
        break
    
    time.sleep(interval)
    elapsed += interval
    interval = min(interval * 1.5, 30)
else:
    print(f"\nTimeout after {max_wait}s")

## View Execution Plan

In [ ]:
resp = requests.get(f"{API_BASE}/api/v1/imagecubes/{cube_id}/plan", headers=headers)
resp.raise_for_status()
plan = resp.json()
print(json.dumps(plan, indent=2))

## List Tasks

In [ ]:
resp = requests.get(f"{API_BASE}/api/v1/imagecubes/{cube_id}/tasks", headers=headers)
resp.raise_for_status()
tasks = resp.json()

if isinstance(tasks, list):
    items = tasks
else:
    items = tasks.get("items", tasks.get("tasks", []))

print(f"Tasks: {len(items)}\n")
for t in items[:10]:
    print(f"  {t.get('id','?')} | status={t.get('status')} | type={t.get('type','?')}")

## Cancel / Resume a Cube

Use `PATCH` with `status: "failed"` to cancel, or `status: "dispatching"` to resume.

In [ ]:
# Cancel a running cube
# resp = requests.patch(f"{API_BASE}/api/v1/imagecubes/{cube_id}", json={"status": "failed"}, headers=headers)
# resp.raise_for_status()
# print("Cancelled:", resp.json())

# Resume a failed cube
# resp = requests.patch(f"{API_BASE}/api/v1/imagecubes/{cube_id}", json={"status": "dispatching"}, headers=headers)
# resp.raise_for_status()
# print("Resumed:", resp.json())

## Retry a Failed Task

In [ ]:
# task_id = "<paste-task-id-here>"
# resp = requests.post(f"{API_BASE}/api/v1/imagecubes/{cube_id}/tasks/{task_id}/retry", headers=headers)
# resp.raise_for_status()
# print("Retried:", resp.json())

## Trigger a New Run

In [ ]:
# resp = requests.post(f"{API_BASE}/api/v1/imagecubes/{cube_id}/runs", headers=headers)
# resp.raise_for_status()
# print("New run:", resp.json())

## Delete a Cube (feature-flagged)

In [ ]:
# resp = requests.delete(f"{API_BASE}/api/v1/imagecubes/{cube_id}", headers=headers)
# resp.raise_for_status()
# print("Deleted:", resp.status_code)

## Get Artifact URL & Open Zarr

In [ ]:
resp = requests.get(f"{API_BASE}/api/v1/imagecubes/{cube_id}/artifact-url", headers=headers)
resp.raise_for_status()
artifact = resp.json()
zarr_url = artifact.get("url") or artifact.get("artifact_url") or artifact.get("href", "")
print(f"Zarr URL: {zarr_url[:80]}...")

In [ ]:
import xarray as xr
import numpy as np

ds = xr.open_zarr(zarr_url, consolidated=True)
print(ds)
print(f"\nVariables: {list(ds.data_vars)}")
print(f"Dims: {dict(ds.dims)}")

## Compute NDVI

In [ ]:
b04 = ds["B04"].compute().values.astype(np.float32)
b08 = ds["B08"].compute().values.astype(np.float32)

with np.errstate(divide="ignore", invalid="ignore"):
    ndvi = (b08 - b04) / (b08 + b04)

ndvi = np.where(np.isfinite(ndvi), ndvi, np.nan)
print(f"NDVI shape: {ndvi.shape}")
print(f"NDVI range: {np.nanmin(ndvi):.3f} to {np.nanmax(ndvi):.3f}")
print(f"Mean NDVI: {np.nanmean(ndvi):.3f}")

## Visualize

In [ ]:
import matplotlib.pyplot as plt

# Plot first timestep
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
if ndvi.ndim == 3:
    im = ax.imshow(ndvi[0], cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    ax.set_title(f"NDVI — {str(ds.coords['time'].values[0])[:10]}")
else:
    im = ax.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    ax.set_title("NDVI")
plt.colorbar(im, ax=ax, label="NDVI")
plt.tight_layout()
plt.show()

## NDVI Time Series (mean per timestep)

In [ ]:
if ndvi.ndim == 3 and "time" in ds.coords:
    means = [np.nanmean(ndvi[t]) for t in range(ndvi.shape[0])]
    times = [str(t)[:10] for t in ds.coords["time"].values]
    
    plt.figure(figsize=(10, 4))
    plt.plot(times, means, "g.-", linewidth=2)
    plt.xlabel("Date")
    plt.ylabel("Mean NDVI")
    plt.title("NDVI Time Series")
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Single timestep — no time series to plot.")